# Sinq syncing

Two-phase overnight batch:

1. **Sync source Sinqs** — the canonical `dataset_*` Sinqs where scalars are actually computed. `sinq.sync()` restores each Table, runs every `compute_*` scalar, saves the row, and drops the Table before moving on. Progress is safe against mid-run crashes.
2. **Rebuild composite Sinqs** — pull rows from the freshly-synced sources via `build_composite_sinq`. Composites are always constructed fresh, so we don't sync them directly — that would duplicate work.

Set `OVERWRITE = True` to re-derive every scalar from scratch (slow, restores every Table). Leave `False` to only fill in missing values (skips fully-populated rows). The composite rebuild always runs — cheap, just merges rows.

In [1]:
from mapd import Sinq
from mapd.sinq_builders import build_composite_sinq
from datetime import datetime
import traceback

%load_ext autoreload
%autoreload 2

## Config

In [2]:
# Set to True to re-derive every scalar from scratch (ignores whether the row
# already has values). Leave False for a normal incremental fill.
OVERWRITE = False

# Source Sinqs — where scalars are actually computed. Comment out to skip.
SOURCE_SINQS = [
    'dataset_1',
    'dataset_2_controls',
    'dataset_3_genetic_manipulations',
    'dataset_4_mns',
]

# Composite Sinqs — rebuilt fresh from source Sinqs after syncing.
# Each entry: {'name': <composite name>, 'sources': [<source Sinq names>]}
COMPOSITE_SINQS = [
    {'name': 'dataset_1_2_pca_lda',
     'sources': ['dataset_1', 'dataset_2_controls']},
    # {'name': 'dataset_all',
    #  'sources': ['dataset_1', 'dataset_2_controls',
    #              'dataset_3_genetic_manipulations', 'dataset_4_mns']},
]

print(f'OVERWRITE = {OVERWRITE}')
print(f'{len(SOURCE_SINQS)} source Sinqs:')
for name in SOURCE_SINQS:
    print(f'  - {name}')
print(f'{len(COMPOSITE_SINQS)} composite Sinqs:')
for spec in COMPOSITE_SINQS:
    print(f"  - {spec['name']}  <-  {spec['sources']}")

OVERWRITE = False
4 source Sinqs:
  - dataset_1
  - dataset_2_controls
  - dataset_3_genetic_manipulations
  - dataset_4_mns
1 composite Sinqs:
  - dataset_1_2_pca_lda  <-  ['dataset_1', 'dataset_2_controls']


## Pre-flight: show what will be touched

Loads each source Sinq and reports how many missing values it has (i.e. how much work `sync()` will actually do when `OVERWRITE=False`). Handy sanity check before kicking off an overnight run.

In [3]:
for name in SOURCE_SINQS:
    try:
        s = Sinq(sinqname=name)
        df_no_table = s.df.drop(columns=['Table', s.NOTES_COLUMN], errors='ignore')
        missing = int(df_no_table.isna().sum().sum())
        rows_with_missing = int(df_no_table.isna().any(axis=1).sum())
        print(f'{name}: {len(s.df)} rows, {missing} missing values across {rows_with_missing} rows')
    except Exception as e:
        print(f'{name}: FAILED to load — {e!r}')

Found data directory: D:\Data
dataset_1: 25 rows, 72 missing values across 24 rows
Found data directory: D:\Data
dataset_2_controls: 39 rows, 114 missing values across 38 rows
Found data directory: D:\Data
dataset_3_genetic_manipulations: 36 rows, 102 missing values across 34 rows
Found data directory: D:\Data
dataset_4_mns: 12 rows, 0 missing values across 0 rows


## Phase 1 — sync source Sinqs

Each source Sinq is synced with its own try/except so a bad Table in one dataset doesn't kill the whole run. Progress is timestamped and the failing traceback (if any) is printed but not re-raised.

In [ ]:
def _stamp():
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S')

source_results = {}
run_started = datetime.now()
print(f'[{_stamp()}] Source sync started (OVERWRITE={OVERWRITE})')

for name in SOURCE_SINQS:
    print(f'\n[{_stamp()}] ====== {name} ======')
    started = datetime.now()
    try:
        s = Sinq(sinqname=name)
        s.sync(overwrite=OVERWRITE)
        source_results[name] = ('ok', datetime.now() - started)
        print(f'[{_stamp()}] {name}: done in {source_results[name][1]}')
    except Exception as e:
        source_results[name] = ('error', str(e))
        print(f'[{_stamp()}] {name}: FAILED — {e!r}')
        traceback.print_exc()

print(f'\n[{_stamp()}] Source sync finished. Elapsed: {datetime.now() - run_started}')

## Phase 2 — rebuild composite Sinqs

Composite Sinqs are built fresh (`fresh=True` inside `build_composite_sinq`) by merging rows from the source Sinqs listed in the config. No scalars are recomputed here — the sources were just synced. If a source failed to sync in Phase 1, its composites will still build but may carry the same missing values.

In [5]:
composite_results = {}
print(f'[{_stamp()}] Composite rebuild started')

for spec in COMPOSITE_SINQS:
    comp_name = spec['name']
    source_names = spec['sources']
    print(f'\n[{_stamp()}] ====== {comp_name} <- {source_names} ======')
    started = datetime.now()
    try:
        sources = [Sinq(sinqname=n) for n in source_names]
        composite = build_composite_sinq(name=comp_name, sources=sources)
        composite_results[comp_name] = ('ok', datetime.now() - started, len(composite.df))
        print(f'[{_stamp()}] {comp_name}: {len(composite.df)} rows, built in {composite_results[comp_name][1]}')
    except Exception as e:
        composite_results[comp_name] = ('error', str(e), None)
        print(f'[{_stamp()}] {comp_name}: FAILED — {e!r}')
        traceback.print_exc()

print(f'\n[{_stamp()}] All done. Total elapsed: {datetime.now() - run_started}')

[2026-07-21 15:20:24] Composite rebuild started

[2026-07-21 15:20:24] ====== dataset_1_2_pca_lda <- ['dataset_1', 'dataset_2_controls'] ======
Found data directory: D:\Data
Found data directory: D:\Data
Found data directory: D:\Data
File D:\Data\Sinqs\dataset_1_2_pca_lda.pkl exists but starting fresh.
[2026-07-21 15:20:25] dataset_1_2_pca_lda: 64 rows, built in 0:00:00.564090

[2026-07-21 15:20:25] All done. Total elapsed: 2:28:41.868053


## Summary

In [6]:
print('Sources:')
for name, (status, detail) in source_results.items():
    print(f'  {name}: {status}  ({detail})')
print('Composites:')
for name, (status, detail, nrows) in composite_results.items():
    extra = f', {nrows} rows' if nrows is not None else ''
    print(f'  {name}: {status}  ({detail}{extra})')

Sources:
  dataset_1: ok  (0:32:42.131061)
  dataset_2_controls: ok  (0:51:13.031768)
  dataset_3_genetic_manipulations: ok  (0:49:47.052462)
  dataset_4_mns: ok  (0:00:47.468680)
Composites:
  dataset_1_2_pca_lda: ok  (0:00:00.564090, 64 rows)
